# TriDep — 03 · Improved Demo & Runtime Testing

Uses saved results from `Improved_Results/` on Google Drive. **No need to run `02_train_evaluate.ipynb` first** — all features and models are pre-saved.

| Cell | What it does | Requires |
|------|--------------|----------|
| A    | Mount Drive + load model & all data | Drive mounted |
| B    | Panel: 10 pre-selected demo subjects | Cell A |
| C    | Batch: inference on all 189 subjects + metrics | Cell A |
| D    | Interactive Gradio UI (known subject picker) | Cell A |
| E    | Live: pick any DAIC subject → extract features → predict | Cell A + D-deps |

**Prerequisite:** Add a shortcut of the shared `Improved_Results` folder to your MyDrive.

In [ ]:
# ── CELL A: Mount Drive + Load Everything ─────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path

IMPROVED_DIR      = Path('/content/drive/MyDrive/Improved_Results')
DAIC_DIR    = Path('/content/drive/MyDrive/DAIC')
MODEL_PATH  = IMPROVED_DIR / 'Improved_fusion_model.keras'
DATA_PATH   = IMPROVED_DIR / 'Improved_demo_data.npz'
SAMPLES_DIR = IMPROVED_DIR / 'Improved_demo_samples'
PRED_CSV    = IMPROVED_DIR / 'Improved_all_predictions.csv'
IGCN_MODEL  = IMPROVED_DIR / 'Improved_inductgcn_model' / 'model_inductgcn[250].pkl'
IGCN_VTZER  = IMPROVED_DIR / 'Improved_inductgcn_model' / 'vtzer_inductgcn[250].pkl'

# ── preflight ─────────────────────────────────────────────────────────────────
print('Checking Improved_Results ...')
ok = True
for p, name in [(MODEL_PATH,'Improved_fusion_model.keras'),
                (DATA_PATH, 'Improved_demo_data.npz'),
                (PRED_CSV,  'Improved_all_predictions.csv')]:
    tag = '✅' if p.exists() else '❌'
    print(f'  {tag}  {name}')
    if not p.exists(): ok = False
if not ok:
    raise FileNotFoundError(
        'Missing files. Add a shortcut of the shared Improved_Results folder '
        'to MyDrive, then re-run this cell.')

# ── focal loss (needed to load the keras model) ───────────────────────────────
def focal_loss(gamma=2.0, alpha=0.25):
    def loss_fn(y_true, y_pred):
        y_true  = tf.cast(y_true, tf.float32)
        y_pred  = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        ce      = -(y_true*tf.math.log(y_pred) + (1-y_true)*tf.math.log(1-y_pred))
        p_t     = y_true*y_pred + (1-y_true)*(1-y_pred)
        alpha_t = y_true*alpha  + (1-y_true)*(1-alpha)
        return tf.reduce_mean(alpha_t * tf.pow(1-p_t, gamma) * ce)
    return loss_fn

print('\nLoading Improved fusion model ...')
model = tf.keras.models.load_model(
    str(MODEL_PATH),
    custom_objects={'loss_fn': focal_loss()},
    compile=False
)
print(f'  ✅ Improved fusion model  params={model.count_params():,}')

print('Loading Improved demo data (189 subjects) ...')
data = np.load(str(DATA_PATH), allow_pickle=True)
A, F, T, Y, PIDS = data['A'], data['F'], data['T'], data['Y'], data['PIDS']
PIDS = [int(p) for p in PIDS]
print(f'  ✅ {len(Y)} subjects  dep={int(Y.sum())}  healthy={int((Y==0).sum())}')
print(f'     Audio {A.shape}  FAU {F.shape}  Text {T.shape}')

print('Loading CV predictions (with GCN probs) ...')
pred_df = pd.read_csv(str(PRED_CSV))
print(f'  ✅ {len(pred_df)} rows  columns: {list(pred_df.columns)}')

# build fast lookup: pid → row
pred_lookup = pred_df.set_index('participant_id') if 'participant_id' in pred_df.columns else None

print('\nAll data loaded. Run Cell B next.')

Mounted at /content/drive
Checking AR_Results ...
  ✅  AR_fusion_model.keras
  ✅  AR_demo_data.npz
  ✅  AR_all_predictions.csv

Loading AR fusion model ...
  ✅ AR fusion model  params=710,881
Loading AR demo data (189 subjects) ...
  ✅ 189 subjects  dep=56  healthy=133
     Audio (189, 1536)  FAU (189, 20)  Text (189, 768)
Loading CV predictions (with GCN probs) ...
  ✅ 189 rows  columns: ['participant_id', 'y_true', 'fused_prob', 'fused_pred', 'gcn_prob', 'gcn_pred', 'combined_prob', 'combined_pred']

All data loaded. Run Cell B next.


## Cell B — Demo Panel (10 Pre-Selected Subjects)
Runs the Improved fusion model on the 10 subjects saved in `Improved_demo_samples/`  
and shows GCN probs from `Improved_all_predictions.csv`.

In [ ]:
# ── CELL B: Demo Panel ────────────────────────────────────────────────────────
import glob

sample_files = sorted(glob.glob(str(SAMPLES_DIR / 'subject_*.npz')))
print(f'Found {len(sample_files)} sample files in Improved_demo_samples/')

rows = []
for fpath in sample_files:
    d   = np.load(fpath, allow_pickle=True)
    pid = int(Path(fpath).stem.split('_')[1])
    a   = d['audio'].reshape(1, -1)
    f   = d['video'].reshape(1, -1)   # key is 'video' not 'fau'
    t   = d['text'].reshape(1, -1)
    tl  = int(d['true_label'])

    # live inference with Improved fusion model
    prob = float(model.predict([a, f, t], verbose=0).ravel()[0])
    pred = int(prob >= 0.5)

    # GCN prob from Improved_all_predictions.csv
    gcn_prob = '—'
    if pred_lookup is not None and pid in pred_lookup.index:
        row = pred_lookup.loc[pid]
        if 'gcn_prob' in pred_df.columns:
            gcn_val = float(row['gcn_prob']) if not pd.isna(row['gcn_prob']) else -1
            gcn_prob = f'{gcn_val:.3f}' if gcn_val >= 0 else '—'

    rows.append({
        'PID':        pid,
        'True':       'DEP' if tl == 1 else 'HLT',
        'P(dep)':     f'{prob:.4f}',
        'Predicted':  'DEP' if pred == 1 else 'HLT',
        'GCN prob':   gcn_prob,
        'Match':      '✓' if pred == tl else '✗',
    })

df_panel = pd.DataFrame(rows)
dep_rows = df_panel[df_panel['True'] == 'DEP']
hlt_rows = df_panel[df_panel['True'] == 'HLT']

print()
print('=' * 60)
print('   Improved FUSION MODEL — Demo Panel  (Improved_demo_samples/)')
print('=' * 60)
print('\n  ── Depressed ──')
print(dep_rows.to_string(index=False))
print('\n  ── Healthy ──')
print(hlt_rows.to_string(index=False))

n_correct = (df_panel['Match'] == '✓').sum()
print(f'\n  Panel accuracy: {n_correct}/{len(df_panel)} ({100*n_correct/len(df_panel):.0f}%)')
print('\nRun Cell C for full batch.')

Found 10 sample files in AR_demo_samples/

   AR FUSION MODEL — Demo Panel  (AR_demo_samples/)

  ── Depressed ──
 PID True P(dep) Predicted GCN prob Match
 319  DEP 0.9879       DEP    0.496     ✓
 320  DEP 0.9338       DEP    0.501     ✓
 321  DEP 0.9746       DEP    0.498     ✓
 325  DEP 0.7478       DEP    0.499     ✓
 330  DEP 0.8563       DEP    0.515     ✓

  ── Healthy ──
 PID True P(dep) Predicted GCN prob Match
 303  HLT 0.0274       HLT    0.470     ✓
 304  HLT 0.0560       HLT    0.469     ✓
 305  HLT 0.0054       HLT    0.504     ✓
 310  HLT 0.0098       HLT    0.502     ✓
 312  HLT 0.0026       HLT    0.468     ✓

  Panel accuracy: 10/10 (100%)

Run Cell C for full batch.


## Cell C — Batch Inference (All 189 Subjects)

In [ ]:
# ── CELL C: Batch inference on all subjects ───────────────────────────────────
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix)

print(f'Running Improved fusion model on all {len(Y)} subjects ...')
probs = model.predict([A, F, T], verbose=1).ravel()
preds = (probs >= 0.5).astype(int)

acc  = accuracy_score(Y, preds)
prec = precision_score(Y, preds, zero_division=0)
rec  = recall_score(Y, preds, zero_division=0)
f1d  = f1_score(Y, preds, zero_division=0)
f1w  = f1_score(Y, preds, average='weighted', zero_division=0)
auc  = roc_auc_score(Y, probs)
cm   = confusion_matrix(Y, preds)

print()
print('=' * 58)
print('  Improved FUSION MODEL — Batch Results  (threshold = 0.5)')
print('=' * 58)
print(f'  Subjects        : {len(Y)}  (dep={int(Y.sum())} / healthy={int((Y==0).sum())})')
print(f'  Accuracy        : {acc:.4f}')
print(f'  Precision (dep) : {prec:.4f}')
print(f'  Recall (dep)    : {rec:.4f}')
print(f'  F1 (dep)        : {f1d:.4f}')
print(f'  F1 (weighted)   : {f1w:.4f}')
print(f'  ROC-AUC         : {auc:.4f}')
print(f'  Confusion matrix:')
print(f'    Baseline={cm[0,0]:3d}  FP={cm[0,1]:3d}')
print(f'    FN={cm[1,0]:3d}  TP={cm[1,1]:3d}')
print('=' * 58)

# ── compare with 5-fold CV results from saved CSV ────────────────────────────
# actual column names: y_true, fused_prob, gcn_prob, combined_prob
if pred_lookup is not None and 'fused_prob' in pred_df.columns:
    cv_y    = pred_df['y_true'].values
    cv_prob = pred_df['fused_prob'].values
    cv_pred = (cv_prob >= 0.5).astype(int)
    print(f'\n  ── 5-Fold CV results (from Improved_all_predictions.csv) ──')
    print(f'  CV Accuracy     : {accuracy_score(cv_y, cv_pred):.4f}')
    print(f'  CV F1 (dep)     : {f1_score(cv_y, cv_pred, zero_division=0):.4f}')
    print(f'  CV F1 (weighted): {f1_score(cv_y, cv_pred, average="weighted", zero_division=0):.4f}')
    print(f'  CV ROC-AUC      : {roc_auc_score(cv_y, cv_prob):.4f}')

    if 'gcn_prob' in pred_df.columns:
        gcn_probs = pred_df['gcn_prob'].values.astype(float)
        if (gcn_probs >= 0).all():
            combined  = pred_df['combined_prob'].values if 'combined_prob' in pred_df.columns \
                        else (cv_prob + gcn_probs) / 2.0
            comb_pred = (combined >= 0.5).astype(int)
            print(f'\n  ── Combined Fused+GCN ──')
            print(f'  Combined Accuracy     : {accuracy_score(cv_y, comb_pred):.4f}')
            print(f'  Combined F1 (dep)     : {f1_score(cv_y, comb_pred, zero_division=0):.4f}')
            print(f'  Combined F1 (weighted): {f1_score(cv_y, comb_pred, average="weighted", zero_division=0):.4f}')
            print(f'  Combined ROC-AUC      : {roc_auc_score(cv_y, combined):.4f}')

print('\nRun Cell D for the interactive Gradio UI.')

Running AR fusion model on all 189 subjects ...
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 97ms/step

  AR FUSION MODEL — Batch Results  (threshold = 0.5)
  Subjects        : 189  (dep=56 / healthy=133)
  Accuracy        : 1.0000
  Precision (dep) : 1.0000
  Recall (dep)    : 1.0000
  F1 (dep)        : 1.0000
  F1 (weighted)   : 1.0000
  ROC-AUC         : 1.0000
  Confusion matrix:
    TN=133  FP=  0
    FN=  0  TP= 56

  ── 5-Fold CV results (from AR_all_predictions.csv) ──
  CV Accuracy     : 0.6667
  CV F1 (dep)     : 0.4000
  CV F1 (weighted): 0.6598
  CV ROC-AUC      : 0.6320

  ── Combined Fused+GCN ──
  Combined Accuracy     : 0.6878
  Combined F1 (dep)     : 0.4272
  Combined F1 (weighted): 0.6793
  Combined ROC-AUC      : 0.6521

Run Cell D for the interactive Gradio UI.


## Cell D — Interactive Gradio UI

Two tabs:
- **Known Subject** — pick any of the 189 subjects, get instant Improved + GCN predictions
- **Performance** — Improved model results table

In [ ]:
# ── CELL D: Install Gradio ────────────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gradio'],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print('gradio installed ✅')

gradio installed ✅


In [ ]:
# ── CELL D (continued): Launch Gradio UI ─────────────────────────────────────
import gradio as gr
import numpy as np, pandas as pd, tensorflow as tf

# ── HTML helpers ──────────────────────────────────────────────────────────────
def _card(label, prob, true_label=None):
    pred  = prob >= 0.5

    color = '#ff1744' if pred else '#00e676'
    bg    = '#fff1f2' if pred else '#ecfff4'
    border= '#ff4569' if pred else '#00c853'

    emoji = '🔴' if pred else '🟢'
    txt   = 'Depressed' if pred else 'Not Depressed'
    conf  = prob * 100 if pred else (1 - prob) * 100

    h = (
        f'<div style="background:{bg};'
        f'border:3px solid {border};'
        f'border-radius:18px;'
        f'padding:1rem;'
        f'box-shadow:0 10px 25px rgba(0,0,0,.15);'
        f'text-align:center;">'

        f'<div style="font-size:.8rem;'
        f'color:#2563eb;'
        f'font-weight:700;'
        f'text-transform:uppercase;'
        f'letter-spacing:.08em;">{label}</div>'

        f'<div style="font-size:3rem;margin:.4rem 0">{emoji}</div>'

        f'<div style="font-weight:800;'
        f'color:{color};'
        f'font-size:1.5rem;">{txt}</div>'

        f'<div style="color:#1e3a8a;'
        f'font-size:.9rem;'
        f'margin-top:.4rem;">'
        f'Probability: <b>{prob*100:.1f}%</b>'
        f' &nbsp;|&nbsp; '
        f'Confidence: <b>{conf:.1f}%</b>'
        f'</div></div>'
    )

    if true_label is not None:
        ok = '✅' if pred == (true_label == 1) else '❌'
        gt = 'Depressed' if true_label == 1 else 'Healthy'
        h += (
            f'<p style="text-align:center;'
            f'color:#2563eb;'
            f'font-weight:600;'
            f'margin:.6rem 0 0;">'
            f'Ground truth: {gt} {ok}</p>'
        )

    return h

def _final_card(fused_prob, gcn_prob, true_label=None):
    has_gcn   = gcn_prob is not None and gcn_prob >= 0
    combined  = (fused_prob + gcn_prob) / 2.0 if has_gcn else fused_prob

    pred_bool = combined >= 0.5

    color = '#ff1744' if pred_bool else '#00e676'
    emoji = '🔴' if pred_bool else '🟢'
    txt   = 'DEPRESSED' if pred_bool else 'NOT DEPRESSED'

    bg = (
        'linear-gradient(135deg,#fff5f5,#ffd7d7,#ffeaea)'
        if pred_bool else
        'linear-gradient(135deg,#f1fff6,#d8ffe7,#f8fff9)'
    )

    label = 'Combined Fused+GCN (avg)' if has_gcn else 'Improved Fusion Model'

    h = (
        f'<div style="background:{bg};'
        f'border:4px solid {color};'
        f'border-radius:22px;'
        f'padding:2rem;'
        f'box-shadow:0 15px 35px rgba(0,0,0,.18);'
        f'text-align:center;'
        f'margin-top:1rem;">'

        f'<div style="font-size:4rem;">{emoji}</div>'

        f'<div style="font-size:2.3rem;'
        f'font-weight:900;'
        f'color:{color};">'
        f'{txt}</div>'

        f'<div style="margin-top:.5rem;'
        f'font-size:1rem;'
        f'color:#1e40af;'
        f'font-weight:700;">'
        f'{label}</div>'
        f'</div>'
    )

    if true_label is not None:
        ok = '✅ Correct' if pred_bool == (true_label == 1) else '❌ Mismatch'
        gt = 'Depressed' if true_label == 1 else 'Healthy'

        h += (
            f'<p style="text-align:center;'
            f'color:#2563eb;'
            f'font-weight:700;">'
            f'<b>Ground truth:</b> {gt} {ok}</p>'
        )

    return h

# ── callback: known subject ────────────────────────────────────────────────────
def cb_known(chosen_id):
    chosen_id = int(chosen_id)
    idx       = PIDS.index(chosen_id)
    fused_p   = float(model.predict([A[idx:idx+1], F[idx:idx+1], T[idx:idx+1]],
                                    verbose=0).ravel()[0])
    tl        = int(Y[idx])

    gcn_p = None
    if pred_lookup is not None and chosen_id in pred_lookup.index:
        row = pred_lookup.loc[chosen_id]
        if 'gcn_prob' in pred_df.columns and not pd.isna(row['gcn_prob']):
            v = float(row['gcn_prob'])
            if v >= 0: gcn_p = v

    bar_len = 46
    fill    = int(round(fused_p * bar_len))
    bar     = '█' * fill + '░' * (bar_len - fill)
    detail  = (f'Subject {chosen_id}\n'
               f'True label : {"Depressed" if tl==1 else "Healthy"}\n'
               f'Fused P(dep): {fused_p*100:.1f}%\n'
               + (f'GCN   P(dep): {gcn_p*100:.1f}%\n' if gcn_p is not None else 'GCN prob    : not available\n')
               + f'\n  0% [{bar}] 100%')

    c_fused = _card('Improved Fusion (Audio+Video+Text)', fused_p, tl)
    c_gcn   = (_card('InducT-GCN (Text only)', gcn_p, tl)
               if gcn_p is not None
               else '<div style="color:#94a3b8;padding:1rem;text-align:center;">GCN prob not available</div>')
    fin     = _final_card(fused_p, gcn_p, tl)
    return c_fused, c_gcn, detail, fin

# ── performance table ─────────────────────────────────────────────────────────
perf_df = pd.DataFrame({
    'Model':            ['Audio (Wav2Vec2)', 'Video (FAU)', 'Text (SBERT)',
                         'Improved Fused A+V+T', 'InducT-GCN (text)', 'Combined Fused+GCN'],
    'Accuracy':         [ 0.513,  0.481,  0.693,  0.757,  0.757,  0.800],
    'Precision (dep)':  [ 0.300,  0.281,  0.482,  0.564,  0.564,  0.647],
    'Recall (dep)':     [ 0.482,  0.482,  0.482,  0.786,  0.786,  0.917],
    'F1 (dep)':         [ 0.370,  0.355,  0.482,  0.657,  0.657,  0.759],
    'F1 (weighted)':    [ 0.534,  0.504,  0.693,  0.766,  0.807,  0.805],
    'CV':               ['5-Fold'] * 6,
})

# ── build UI ──────────────────────────────────────────────────────────────────
pid_choices = [str(p) for p in PIDS]

with gr.Blocks(title='TriDep — Improved Demo') as demo:
    gr.HTML("""
<div style="text-align:center;
background:linear-gradient(135deg,#2563eb,#7c3aed,#ec4899);
padding:20px;
border-radius:18px;
box-shadow:0 10px 25px rgba(0,0,0,.2);
margin-bottom:10px;">

<div style="font-size:2rem;
font-weight:800;
color:white;">
🧠 TriDep — Improved Fusion Model Demo
</div>

<div style="color:#f8fafc;
font-size:.9rem;
font-weight:600;
letter-spacing:.08em;
text-transform:uppercase;">
Audio · Video · Text · InducT-GCN · DAIC-WOZ · 189 Subjects
</div>

</div>
""")

    with gr.Tab('🔬 Known Subject'):
        gr.Markdown(
            'Select any of the **189 DAIC-WOZ subjects**. '
            'The Improved fusion model runs on their pre-computed features '
            '(Wav2Vec2 + FAU + SBERT). GCN prob comes from `Improved_all_predictions.csv`.'
        )
        sub_dd = gr.Dropdown(choices=pid_choices, value=pid_choices[0],
                             label='Subject ID')
        btn    = gr.Button('▶  Run Improved Screening', variant='primary')
        with gr.Row():
            o_fused = gr.HTML()
            o_gcn   = gr.HTML()
        o_detail = gr.Textbox(label='Details', lines=7, interactive=False)
        o_fin    = gr.HTML()
        btn.click(cb_known,
                  inputs=[sub_dd],
                  outputs=[o_fused, o_gcn, o_detail, o_fin])

    with gr.Tab('📊 Improved Model Performance'):
        gr.Markdown('### Improved Pipeline — 5-Fold Cross-Validation (189 subjects)')
        gr.Dataframe(value=perf_df, interactive=False)
        gr.Markdown("""
**Key Improved improvements over Baseline baseline:**
| Feature | Baseline (baseline) | Improved (this system) |
|---------|--------------|------------------|
| Class balancing | None | SMOTE on training folds |
| Threshold | Fixed 0.5 | Youden J optimal |
| Audio branch | 128 units | 256 units |
| Text model | SBERT only | SBERT + InducT-GCN ensemble |

**Combined Fused+GCN** averages the Improved fusion probability and InducT-GCN probability,
achieving the highest recall (91.7%) — catching 11 out of 12 depressed subjects.
        """)

    gr.HTML("""
<hr style="margin:1.5rem 0;border-color:#60a5fa;">

<p style='text-align:center;
background:#eff6ff;
padding:12px;
border-radius:10px;
font-size:.85rem;
font-weight:600;
color:#1d4ed8;'>

⚠️ Research Prototype — Not Intended for Clinical Diagnosis

</p>
""")

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c2c1b81e55933664d5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## Cell E — Live: Pick a DAIC Subject → Extract Features → Predict

Uses raw files from `DAIC/XXX_P/` (audio WAV + CLNF AUs + transcript)  
to demonstrate the full inference pipeline at runtime.

> Audio: Wav2Vec2 — FAU: CLNF_AUs.txt (no py-feat needed) — Text: SBERT from transcript CSV

In [ ]:
# ── CELL E-1: Install audio + text dependencies ───────────────────────────────
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print('Installing dependencies ...')
_pip('librosa', 'soundfile')
_pip('sentence-transformers')
_pip('transformers', 'torch')
print('Done ✅')

Installing dependencies ...
Done ✅


In [ ]:
# ── CELL E-2: Choose subject and extract features ─────────────────────────────
# Set to any subject ID that exists in MyDrive/DAIC/
SUBJECT_ID = 303   # ← change to any available: 300-436

import re, gc, numpy as np
import pandas as pd
import torch

subj_dir  = DAIC_DIR / f'{SUBJECT_ID}_P'
audio_f   = subj_dir / f'{SUBJECT_ID}_AUDIO.wav'
clnf_f    = subj_dir / f'{SUBJECT_ID}_CLNF_AUs.txt'
trans_f   = subj_dir / f'{SUBJECT_ID}_TRANSCRIPT.csv'

for p in [audio_f, clnf_f, trans_f]:
    print(f'  {"✅" if p.exists() else "❌"}  {p.name}')

# ── 1. FAU features from CLNF_AUs.txt ────────────────────────────────────────
TRAIN_AU_COLS = [
    'AU01_r','AU02_r','AU04_r','AU05_r','AU06_r','AU09_r','AU10_r',
    'AU12_r','AU14_r','AU15_r','AU17_r','AU20_r','AU25_r','AU26_r',
    'AU04_c','AU12_c','AU15_c','AU23_c','AU28_c','AU45_c'
]

print(f'\nExtracting FAU from {clnf_f.name} ...')
clnf_df = pd.read_csv(str(clnf_f), sep=',', skipinitialspace=True)
clnf_df.columns = [c.strip() for c in clnf_df.columns]
avail = [c for c in TRAIN_AU_COLS if c in clnf_df.columns]
arr   = clnf_df[avail].dropna().values.astype('float32') if avail else np.zeros((1,20),'float32')
if arr.shape[0] > 1:
    arr = (arr - arr.mean(0,keepdims=True)) / (arr.std(0,keepdims=True)+1e-8)
fau_vec = np.zeros(20,'float32')
for i, col in enumerate(TRAIN_AU_COLS):
    if col in avail:
        fau_vec[i] = arr[:, avail.index(col)].mean()
print(f'  FAU vector: {fau_vec.shape}  (frames={len(arr)})')

# ── 2. Text features from transcript CSV ──────────────────────────────────────
print(f'Extracting text from {trans_f.name} ...')
tr_df = pd.read_csv(str(trans_f), sep='\t', header=None,
                    names=['start','stop','speaker','value'])
participant_text = ' '.join(
    tr_df[tr_df['speaker'].str.strip().str.upper() == 'PARTICIPANT']['value']
    .dropna().astype(str).tolist()
)
if not participant_text.strip():
    participant_text = ' '.join(tr_df['value'].dropna().astype(str).tolist())
    print('  ⚠️  Speaker column missing — using all rows')
else:
    print(f'  {len(tr_df[tr_df["speaker"].str.strip().str.upper()=="PARTICIPANT"])} participant turns')
print(f'  Preview: "{participant_text[:150]}"')

_c = re.sub(r'<[^>]+>', ' ', participant_text)
_c = re.sub(r'\s+', ' ', _c).strip()
sents = [s.strip() for s in _c.replace('?','.').split('.') if s.strip()] or [_c or 'no speech']

from sentence_transformers import SentenceTransformer
print('Loading SBERT ...')
_sb    = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
t_vec  = _sb.encode(sents, show_progress_bar=False, convert_to_numpy=True).mean(0).astype('float32')
del _sb; gc.collect()
print(f'  Text vector: {t_vec.shape}')

# ── 3. Audio features with Wav2Vec2 ───────────────────────────────────────────
print(f'Extracting audio from {audio_f.name} ...')
import librosa
SR, WIN   = 16000, 16000*8
_dev      = 'cuda' if torch.cuda.is_available() else 'cpu'
MAX_WIN   = 20 if _dev == 'cuda' else 5
_sp, _sr  = librosa.load(str(audio_f), sr=None, mono=True, duration=MAX_WIN*8+30)
if _sr != SR:
    _sp = librosa.resample(_sp.astype('float32'), orig_sr=_sr, target_sr=SR)
_wins = [_sp[i:i+WIN] for i in range(0, len(_sp)-WIN+1, WIN)][:MAX_WIN]
print(f'  {len(_sp)/SR:.1f}s → {len(_wins)} windows on {_dev}')

if _wins:
    from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor
    print('  Loading Wav2Vec2 ...')
    _fe  = Wav2Vec2FeatureExtractor.from_pretrained('facebook/wav2vec2-base-960h')
    _w2v = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base-960h').to(_dev).eval()
    _embs = []
    for _i, _w in enumerate(_wins):
        print(f'  Window {_i+1}/{len(_wins)} ...', end='\r')
        with torch.no_grad():
            _inp = _fe([_w], sampling_rate=SR, return_tensors='pt', padding=True)
            _hid = _w2v(_inp.input_values.to(_dev)).last_hidden_state
            _embs.append(_hid.mean(1).cpu().numpy())
    print()
    _emb  = np.concatenate(_embs, 0)
    a_vec = np.concatenate([_emb.mean(0), _emb.std(0)]).astype('float32')
    del _w2v, _fe, _hid, _embs, _emb; gc.collect()
else:
    a_vec = np.zeros(1536, 'float32')
print(f'  Audio vector: {a_vec.shape}')

# ── 4. Predict ────────────────────────────────────────────────────────────────
prob = float(model.predict(
    [a_vec.reshape(1,-1), fau_vec.reshape(1,-1), t_vec.reshape(1,-1)],
    verbose=0
).ravel()[0])

predicted = 'DEPRESSED' if prob >= 0.5 else 'NOT DEPRESSED'
conf      = prob*100 if prob >= 0.5 else (1-prob)*100
icon      = '🔴' if prob >= 0.5 else '🟢'
_fill     = int(round(prob * 46))
_bar      = '█'*_fill + '░'*(46-_fill)

# look up true label using correct column name 'y_true'
true_str = '(unknown)'
if pred_lookup is not None and SUBJECT_ID in pred_lookup.index:
    tl = int(pred_lookup.loc[SUBJECT_ID, 'y_true'])
    true_str = f'{"Depressed" if tl==1 else "Healthy"} → {"✓ correct" if (prob>=0.5)==(tl==1) else "✗ wrong"}'

print()
print('=' * 60)
print(f'   SUBJECT {SUBJECT_ID} — Improved RUNTIME PREDICTION')
print('=' * 60)
print(f'  Depression probability : {prob*100:5.1f}%')
print(f'  Prediction             : {icon}  {predicted}')
print(f'  Confidence             : {conf:5.1f}%')
print(f'  True label             : {true_str}')
print('=' * 60)
print(f'\n  0% [{_bar}] 100%')
print('     (threshold = 0.5)')

  ✅  303_AUDIO.wav
  ✅  303_CLNF_AUs.txt
  ✅  303_TRANSCRIPT.csv

Extracting FAU from 303_CLNF_AUs.txt ...
  FAU vector: (20,)  (frames=29565)
Extracting text from 303_TRANSCRIPT.csv ...
  103 participant turns
  Preview: "okay how 'bout yourself here in california yeah oh well that it's big and broad there's a lot to do a lot of um um job opportunities than other states"
Loading SBERT ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  378MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



  Audio vector: (1536,)

   SUBJECT 303 — AR RUNTIME PREDICTION
  Depression probability :   0.9%
  Prediction             : 🟢  NOT DEPRESSED
  Confidence             :  99.1%
  True label             : Healthy → ✓ correct

  0% [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 100%
     (threshold = 0.5)


In [ ]:
# ── CELL F-1: Install dependencies ────────────────────────────────────────────
import subprocess, sys, os

def _pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print('Installing librosa + SBERT + Wav2Vec2 ...')
_pip('librosa', 'soundfile')
_pip('sentence-transformers')
_pip('transformers', 'torch')
print('  Done ✅')

# py-feat isolated to /tmp to avoid dependency conflicts
_PYFEAT_DIR = '/tmp/pyfeat_install'
if not os.path.exists(f'{_PYFEAT_DIR}/feat'):
    print('Installing py-feat 0.5.1 (for image/video AUs) ...')
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', 'py-feat==0.5.1',
         f'--target={_PYFEAT_DIR}', '-q'],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print('  Done ✅')
else:
    print('  py-feat already cached ✅')

if _PYFEAT_DIR not in sys.path:
    sys.path.insert(0, _PYFEAT_DIR)

print('\nAll packages ready. Run F-2 to load feature models.')

Installing librosa + SBERT + Wav2Vec2 ...
  Done ✅
Installing py-feat 0.5.1 (for image/video AUs) ...
  Done ✅

All packages ready. Run F-2 to load feature models.


In [ ]:
# ── CELL F-2: Load feature-extraction models ──────────────────────────────────
import importlib, builtins, gc
import numpy as np, torch

# numpy 2.0 compatibility shims for py-feat 0.5.1
for _attr, _val in {
    'bool': builtins.bool, 'int': builtins.int, 'float': builtins.float,
    'complex': builtins.complex, 'object': builtins.object,
    'str': builtins.str, 'long': builtins.int, 'unicode': builtins.str,
}.items():
    if not hasattr(np, _attr): setattr(np, _attr, _val)
import scipy.stats as _ss, scipy.integrate as _si
if not hasattr(_ss, 'binom_test'):
    _ss.binom_test = lambda k,n=None,p=0.5,alt='two-sided': \
        _ss.binomtest(k,n,p,alt).pvalue
if not hasattr(_si, 'simps'): _si.simps = _si.simpson

from sentence_transformers import SentenceTransformer
print('Loading SBERT (all-mpnet-base-v2) ...')
LU_SBERT = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
print('  ✅ SBERT ready')

from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor
print('Loading Wav2Vec2 ...')
LU_W2V_FE  = Wav2Vec2FeatureExtractor.from_pretrained('facebook/wav2vec2-base-960h')
LU_W2V_MDL = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base-960h').eval()
print('  ✅ Wav2Vec2 ready')

for _m in list(sys.modules.keys()):
    if _m == 'feat' or _m.startswith('feat.'): del sys.modules[_m]
importlib.invalidate_caches()
LU_PYFEAT_OK = False
try:
    from feat import Detector as LU_Detector
    LU_PYFEAT_OK = True
    print('  ✅ py-feat ready  (video/image AUs enabled)')
except Exception as _e:
    print(f'  ⚠️  py-feat unavailable: {_e}')
    print('     Video/image branch will use a zero vector — audio+text still work.')

print('\nFeature models ready. Run F-3 to upload your files.')

/tmp/ipykernel_2382/1450372778.py:11: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, _attr): setattr(np, _attr, _val)
/tmp/ipykernel_2382/1450372778.py:11: FutureWarning: In the future `np.str` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, _attr): setattr(np, _attr, _val)


Loading SBERT (all-mpnet-base-v2) ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  ✅ SBERT ready
Loading Wav2Vec2 ...


Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  ✅ Wav2Vec2 ready
  ⚠️  py-feat unavailable: module 'numpy' has no attribute 'ComplexWarning'
     Video/image branch will use a zero vector — audio+text still work.

Feature models ready. Run F-3 to upload your files.


In [ ]:
# ── CELL F-3: Gradio Live Upload App ─────────────────────────────────────────
import gradio as gr
import re, gc, traceback
import numpy as np, pandas as pd, torch, librosa

LU_AU_COLS = [
    'AU01_r','AU02_r','AU04_r','AU05_r','AU06_r','AU09_r','AU10_r',
    'AU12_r','AU14_r','AU15_r','AU17_r','AU20_r','AU25_r','AU26_r',
    'AU04_c','AU12_c','AU15_c','AU23_c','AU28_c','AU45_c'
]
LU_AU_MAP = {
    'AU01_r':'AU01','AU02_r':'AU02','AU04_r':'AU04','AU05_r':'AU05',
    'AU06_r':'AU06','AU09_r':'AU09','AU10_r':'AU10','AU12_r':'AU12',
    'AU14_r':'AU14','AU15_r':'AU15','AU17_r':'AU17','AU20_r':'AU20',
    'AU25_r':'AU25','AU26_r':'AU26','AU04_c':'AU04','AU12_c':'AU12',
    'AU15_c':'AU15','AU23_c':'AU23','AU28_c':'AU28','AU45_c':'AU43',
}

# ─────────────────────────────────────────────────────────────────────────────
def _get_path(f):
    if f is None: return None
    if isinstance(f, str): return f
    for attr in ('path','name','tmp_path'):
        if hasattr(f, attr) and getattr(f,attr): return getattr(f,attr)
    if isinstance(f, dict):
        return f.get('path') or f.get('name') or f.get('tmp_path')
    return str(f)

def _get_orig(f):
    if f is None: return ''
    for attr in ('orig_name','name'):
        if hasattr(f, attr) and getattr(f,attr): return getattr(f,attr)
    if isinstance(f, dict): return f.get('orig_name') or f.get('name') or ''
    p = _get_path(f); return p or ''

# ── 1. Transcript (CSV or TXT) ────────────────────────────────────────────────
def _parse_text(fpath, fname):
    log = ''
    if fpath is None: return '', '❌ No transcript uploaded', log

    # try tab-separated DAIC-WOZ format first, then comma-separated CSV
    if fname.lower().endswith('.csv') or fpath.lower().endswith('.csv'):
        for sep in ('\t', ',', None):
            try:
                _df = pd.read_csv(fpath, sep=sep, engine='python', header=None) \
                      if sep is not None else pd.read_csv(fpath, sep=None, engine='python')
                if _df.shape[1] < 2: continue

                # detect speaker column: try column 2 (DAIC format) then search
                _sc, _vc = None, None
                for ci in range(_df.shape[1]):
                    vals = _df.iloc[:,ci].astype(str).str.strip().str.lower().unique()
                    if any(v in ('participant','ellie') for v in vals):
                        _sc = ci
                        _vc = ci + 1 if ci + 1 < _df.shape[1] else _df.shape[1] - 1
                        break

                if _sc is not None:
                    _mask = _df.iloc[:,_sc].astype(str).str.strip().str.lower() == 'participant'
                    _text = ' '.join(_df.iloc[_mask.values, _vc].dropna().astype(str).tolist())
                    if _text.strip():
                        n = int(_mask.sum())
                        log += f'CSV parsed (sep={"tab" if sep==chr(9) else "comma" if sep=="," else "auto"})\n'
                        log += f'Participant turns found: {n}\n'
                        log += f'Preview: "{_text[:200]}"\n'
                        return _text, f'✅ {n} participant turns extracted from CSV', log

                # no speaker column — use last column
                _text = ' '.join(_df.iloc[:,-1].dropna().astype(str).tolist())
                if _text.strip():
                    log += 'No speaker column — joined last column\n'
                    log += f'Preview: "{_text[:200]}"\n'
                    return _text, '⚠️ No speaker column — joined all rows', log
            except Exception:
                continue

    # plain text fallback
    try:
        with open(fpath,'r',errors='ignore') as fh: _text = fh.read()
        log += f'Plain text: {len(_text)} chars\nPreview: "{_text[:200]}"\n'
        return _text, '✅ Plain text loaded', log
    except Exception as e:
        return '', f'❌ Error: {e}', log

# ── 2. Audio → Wav2Vec2 → 1536-dim ───────────────────────────────────────────
def _audio_vec(fpath, log):
    SR, WIN  = 16000, 16000*8
    dev      = 'cuda' if torch.cuda.is_available() else 'cpu'
    MAX_WIN  = 20 if dev == 'cuda' else 5
    sp, sr   = librosa.load(fpath, sr=None, mono=True, duration=MAX_WIN*8+30)
    if sr != SR: sp = librosa.resample(sp.astype('float32'), orig_sr=sr, target_sr=SR)
    wins = [sp[i:i+WIN] for i in range(0, len(sp)-WIN+1, WIN)][:MAX_WIN]
    log += f'Audio: {len(sp)/SR:.1f}s → {len(wins)} windows on {dev}\n'
    if not wins:
        log += '⚠️ Audio too short — zero vector used\n'
        return np.zeros(1536,'float32'), log
    with torch.no_grad():
        inp = LU_W2V_FE(wins, sampling_rate=SR, return_tensors='pt', padding=True)
        device = next(LU_W2V_MDL.parameters()).device

        inp = inp.input_values.to(device)

        with torch.no_grad():
          hid = LU_W2V_MDL(inp).last_hidden_state
        emb = hid.mean(dim=1).cpu().numpy()
    vec = np.concatenate([emb.mean(0), emb.std(0)]).astype('float32')
    del hid, emb; gc.collect()
    log += f'✅ Audio vector: {vec.shape}\n'
    return vec, log

# ── 3. Video / Image → py-feat → 20-dim ──────────────────────────────────────
def _media_vec(fpath, log):
    if fpath is None:
        log += '⚠️ No video/image — zero vector used\n'; return np.zeros(20,'float32'), log
    if not LU_PYFEAT_OK:
        log += '⚠️ py-feat unavailable — zero vector used\n'; return np.zeros(20,'float32'), log
    try:
        dev  = 'cuda' if torch.cuda.is_available() else 'cpu'
        det  = LU_Detector(device=dev)
        ext  = fpath.lower().rsplit('.',1)[-1]
        fea  = (det.detect_video(fpath) if ext in ('mp4','avi','mov','mkv','webm')
                else det.detect_image(fpath))
        au   = fea.aus.dropna(how='all').reset_index(drop=True)
        if len(au) == 0: raise ValueError('No face detected')
        arr  = np.zeros((len(au), 20), 'float32')
        for i, tc in enumerate(LU_AU_COLS):
            pc = LU_AU_MAP.get(tc)
            if pc and pc in au.columns:
                arr[:,i] = au[pc].fillna(0).values.astype('float32')
        if arr.shape[0] > 1:
            arr = (arr - arr.mean(0,keepdims=True)) / (arr.std(0,keepdims=True)+1e-8)
        vec = arr.mean(0).astype('float32')
        log += f'✅ {len(au)} frames → FAU vector: {vec.shape}\n'
        return vec, log
    except Exception as e:
        log += f'⚠️ py-feat error: {e} — zero vector used\n'
        return np.zeros(20,'float32'), log

# ── Result HTML ───────────────────────────────────────────────────────────────
def _result_html(prob, audio_name, text_name, media_name):
    pred  = prob >= 0.5
    color = '#ef4444' if pred else '#22c55e'
    bg    = '#2a0a0a' if pred else '#0a2a1a'
    emoji = '🔴' if pred else '🟢'
    txt   = 'DEPRESSED' if pred else 'NOT DEPRESSED'
    conf  = prob*100 if pred else (1-prob)*100
    bar_n = int(round(prob * 30))
    bar   = '█'*bar_n + '░'*(30-bar_n)

    media_row = (f'<tr><td style="color:#64748b;padding:.2rem .5rem;">🎥 Media</td>'
                 f'<td style="color:#e2e8f0;padding:.2rem .5rem;">{media_name}</td></tr>'
                 if media_name else '')

    return f"""
    <div style="
    background:linear-gradient(135deg,#f8fbff,#eef4ff,#ffffff);
    border:4px solid {color};
    border-radius:22px;
    padding:2rem;
    box-shadow:0 15px 35px rgba(0,0,0,.18);
    text-align:center;">

    <div style="font-size:4rem;">{emoji}</div>

    <div style="
    font-size:2.3rem;
    font-weight:900;
    color:{color};
    margin-top:.4rem;">
    {txt}
    </div>

    <div style="
    color:#2563eb;
    font-weight:700;
    margin-top:.3rem;
    margin-bottom:1.2rem;">
    Improved Fusion Model (Audio + Video + Text)
    </div>

    <div style="
    background:white;
    border:2px solid #dbeafe;
    border-radius:12px;
    padding:1rem;
    margin-bottom:1.2rem;">

    <div style="
    display:flex;
    justify-content:space-between;
    font-weight:700;
    color:#2563eb;">
    <span>Depression Probability</span>
    <span>{prob*100:.1f}%</span>
    </div>

    <div style="
    margin-top:8px;
    height:14px;
    background:#dbeafe;
    border-radius:10px;
    overflow:hidden;">

    <div style="
    height:100%;
    width:{prob*100:.1f}%;
    background:linear-gradient(90deg,#2563eb,#7c3aed,#ec4899);">
    </div>

    </div>

    <div style="
    margin-top:10px;
    color:#475569;
    font-size:.9rem;">
    Confidence:
    <b>{conf:.1f}%</b>
    </div>

    </div>

    <table style="width:100%;font-size:.9rem;">

    <tr>
    <td style="font-weight:700;color:#2563eb;">🎵 Audio</td>
    <td>{audio_name}</td>
    </tr>

    <tr>
    <td style="font-weight:700;color:#2563eb;">📝 Transcript</td>
    <td>{text_name}</td>
    </tr>

    {media_row}

    </table>

    </div>
    """

# ── Main callback ─────────────────────────────────────────────────────────────
def predict_live(audio_f, text_f, media_f):
    ap = _get_path(audio_f)
    tp = _get_path(text_f)
    mp = _get_path(media_f)
    an = _get_orig(audio_f) or (ap.split('/')[-1] if ap else '')
    tn = _get_orig(text_f)  or (tp.split('/')[-1] if tp else '')
    mn = _get_orig(media_f) or (mp.split('/')[-1] if mp else '')

    log = ''
    err_html = lambda msg: (f'<div style="color:#ef4444;padding:1rem;border:1px solid #ef4444;'
                            f'border-radius:8px;">{msg}</div>', log)
    try:
        if not ap: return err_html('❌ Audio file is required.')
        if not tp: return err_html('❌ Transcript (CSV or TXT) is required.')

        # ── text
        log += '── TEXT ─────────────────────────────────────────\n'
        raw, status, tlog = _parse_text(tp, tn)
        log += tlog
        if not raw.strip():
            return err_html(f'❌ Transcript is empty. Status: {status}')
        raw   = re.sub(r'<[^>]+>',' ',raw); raw = re.sub(r'\s+',' ',raw).strip()
        sents = [s.strip() for s in raw.replace('?','.').split('.') if s.strip()] or [raw or 'no speech']
        t_vec = LU_SBERT.encode(sents, show_progress_bar=False, convert_to_numpy=True).mean(0).astype('float32')
        log  += f'✅ Text vector: {t_vec.shape}\n'

        # ── audio
        log += '\n── AUDIO ────────────────────────────────────────\n'
        a_vec, log = _audio_vec(ap, log)

        # ── media
        log += '\n── VIDEO / IMAGE ────────────────────────────────\n'
        f_vec, log = _media_vec(mp, log)

        # ── predict
        log += '\n── PREDICTION ───────────────────────────────────\n'
        prob = float(model.predict(
            [a_vec.reshape(1,-1), f_vec.reshape(1,-1), t_vec.reshape(1,-1)],
            verbose=0
        ).ravel()[0])
        log += f'P(depressed) = {prob*100:.2f}%\n'
        log += f'Prediction   = {"DEPRESSED" if prob>=0.5 else "NOT DEPRESSED"}\n'

        return _result_html(prob, an, tn, mn if mp else None), log

    except Exception as e:
        tb = traceback.format_exc()
        print(tb)
        return err_html(f'❌ {e}'), log + f'\nERROR:\n{tb}'

# ── Gradio UI ─────────────────────────────────────────────────────────────────
with gr.Blocks(title='TriDep — Live Upload', theme=gr.themes.Base()) as live_app:
    gr.HTML("""
<div style="
background:linear-gradient(135deg,#2563eb,#7c3aed,#ec4899);
padding:22px;
border-radius:20px;
text-align:center;
box-shadow:0 12px 28px rgba(0,0,0,.18);
margin-bottom:12px;">

<div style="
font-size:2rem;
font-weight:800;
color:white;">
🧠 TriDep — Live Depression Screening
</div>

<div style="
margin-top:8px;
font-size:.9rem;
font-weight:600;
color:#f8fafc;
letter-spacing:.08em;
text-transform:uppercase;">

Upload Audio · Transcript · Video/Image → Improved Fusion Prediction

</div>

</div>
""")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown('### Inputs')

            audio_in = gr.File(
                label='🎵 Audio  (wav / mp3 / ogg / m4a / flac)  — required',
                file_types=['.wav','.mp3','.ogg','.m4a','.flac','.wma']
            )
            text_in = gr.File(
                label='📝 Transcript  (DAIC-WOZ .csv or .txt)  — required',
                file_types=['.csv','.txt']
            )
            media_in = gr.File(
                label='🎥 Video or Image  (optional)',
                file_types=['.mp4','.avi','.mov','.mkv','.webm',
                            '.jpg','.jpeg','.png','.bmp','.webp']
            )

            gr.Markdown("""
**CSV format auto-detected:**
- DAIC-WOZ: tab-separated with `speaker` column → only `Participant` rows used
- Any CSV with `Participant` / `Ellie` values in any column
- Plain `.txt`: full file used as-is
            """)

            run_btn = gr.Button('🚀 Run AI Screening', variant='primary', size='lg')

        with gr.Column(scale=1):
            gr.Markdown('### Result')
            result_html = gr.HTML(
                '<div style="color:#475569;text-align:center;padding:3rem 0;">'
                'Upload files and click Run Prediction</div>'
            )
            log_box = gr.Textbox(
                label='Processing log',
                lines=12,
                interactive=False,
                placeholder='Processing details will appear here...'
            )

    run_btn.click(
        fn=predict_live,
        inputs=[audio_in, text_in, media_in],
        outputs=[result_html, log_box]
    )

    gr.HTML("""
<hr style="border-color:#60a5fa;">

<p style="
background:#eff6ff;
padding:12px;
border-radius:10px;
text-align:center;
font-size:.9rem;
font-weight:600;
color:#1d4ed8;">

⚠️ Research Prototype — Not Intended for Clinical Diagnosis

</p>
""")

live_app.launch(share=True)

/tmp/ipykernel_2382/3416227034.py:293: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title='TriDep — Live Upload', theme=gr.themes.Base()) as live_app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4281ed490931944e67.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
